<a href="https://colab.research.google.com/github/mrhapile/StaleMind/blob/main/StaleMind_GRPO_Training.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# StaleMind GRPO Training — Llama-3.2-3B-Instruct

**Goal:** Fine-tune Llama-3.2-3B using Group Relative Policy Optimization (GRPO) so it learns to detect preference drift in StaleMind episodes and pick the correct action.

**Budget:** Designed for HuggingFace Spaces / AutoTrain on an A10G (~$1.20/hr). Full run ≈ 2–3 hrs = ~$3–4 per experiment.

**Key fix over eval notebook:** The model's actual output drives action selection — not a hardcoded regex fallback.

## 0. Install Dependencies

In [2]:
!pip install -q trl transformers peft accelerate bitsandbytes datasets
!pip install -q torch --index-url https://download.pytorch.org/whl/cu121

# Clone StaleMind repo — replace with your actual repo URL
!git clone https://huggingface.co/spaces/mrhapile/stalemind /content/stalemind

import sys
sys.path.insert(0, '/content/stalemind')

Cloning into '/content/stalemind'...
remote: Enumerating objects: 58, done.
remote: Counting objects: 100% (54/54), done.
remote: Compressing objects: 100% (47/47), done.
remote: Total 58 (delta 19), reused 0 (delta 0), pack-reused 4 (from 1)
Receiving objects: 100% (58/58), 68.21 KiB | 4.26 MiB/s, done.
Resolving deltas: 100% (19/19), done.


In [3]:
from env.environment import StaleMindEnv

env = StaleMindEnv()
print("ENV LOADED SUCCESSFULLY")

ENV LOADED SUCCESSFULLY


## 1. Direct Environment Interface (No HTTP)

This replaces the `requests.post` pattern from the eval notebook. We import `StaleMindEnv` directly for zero network overhead during training.

In [ ]:
# ── Direct Python interface to StaleMindEnv ──────────────────────────────────
# Adjust the import path to match your repo structure
from env.environment import StaleMindEnv

VALID_ACTIONS = ["ACCEPT", "REJECT", "PROPOSE_RESCHEDULE", "DELEGATE", "PARTIAL_ACCEPT", "IGNORE"]
DRIFT_KEYWORDS = ["son", "family", "needs you", "event today", "emergency", "child"]

def format_obs_as_prompt(obs: dict, step_num: int) -> str:
    """Convert a StaleMind observation dict into a model prompt."""
    visible_prefs = obs.get('visible_preferences', ['work > family'])
    if isinstance(visible_prefs, list):
        visible_prefs = ', '.join(visible_prefs)
    relationships = obs.get('relationships', {})
    rel_str = ', '.join([f"{k}: {v:.2f}" for k, v in relationships.items()]) if relationships else 'unknown'

    return (
        f"You are an AI assistant managing a user's schedule.\n"
        f"Current stated preferences: {visible_prefs}\n"
        f"Relationship health — {rel_str}\n"
        f"Step {step_num}/10: {obs.get('message', '')}\n\n"
        f"Choose exactly one action from: {', '.join(VALID_ACTIONS)}\n"
        f"Respond with JSON: {{\"action\": \"<ACTION>\", \"reasoning\": \"<why>\"}}"
    )

def parse_action_from_completion(text: str) -> str:
    """Extract action from model output. Falls back to ACCEPT only if truly unparseable."""
    import json, re
    text = text.strip()
    # Try JSON parse first
    try:
        # Find JSON block anywhere in the output
        match = re.search(r'\{.*?\}', text, re.DOTALL)
        if match:
            parsed = json.loads(match.group())
            action = parsed.get('action', '').upper().strip()
            if action in VALID_ACTIONS:
                return action
    except Exception:
        pass
    # Fallback: scan output for any valid action keyword
    text_upper = text.upper()
    for action in VALID_ACTIONS:
        if action in text_upper:
            return action
    return "ACCEPT"  # last resort only

def run_episode(model_fn, scenario_index: int = 0) -> float:
    """
    Run a full 10-step StaleMind episode using model_fn(prompt) -> completion.
    Returns total episode reward.
    """
    env = StaleMindEnv(scenario_index=scenario_index)
    obs = env.reset()
    total_reward = 0.0
    for step_num in range(1, 11):
        prompt = format_obs_as_prompt(obs, step_num)
        completion = model_fn(prompt)
        action = parse_action_from_completion(completion)
        obs, reward, done, _ = env.step({"type": action, "content": ""})
        if isinstance(reward, dict):
            reward = reward.get("score", 0.0)
        total_reward += float(reward)
        if done:
            break
    return total_reward

print("Environment interface ready.")

## 2. Load Model with QLoRA (4-bit)

QLoRA lets us fine-tune Llama-3.2-3B on a single A10G (24GB VRAM). Without it, 3B in fp16 alone would use ~6GB just for weights, leaving little room for activations + gradients.

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, TaskType

MODEL_ID = "meta-llama/Llama-3.2-3B-Instruct"

# ── 4-bit quantization config ─────────────────────────────────────────────────
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "left"  # Required for causal LMs during training

base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
)

# ── LoRA adapter config ───────────────────────────────────────────────────────
lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=16,                    # Rank — higher = more expressive but more memory
    lora_alpha=32,           # Scaling factor
    lora_dropout=0.05,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
    bias="none",
)

model = get_peft_model(base_model, lora_config)
model.print_trainable_parameters()
# Expected output: ~1-2% of params trainable — that's correct for LoRA

## 3. Build Training Dataset

GRPO needs a dataset of prompts. We generate these by rolling out StaleMind episodes across all scenarios and capturing the observation at each step as a training prompt.

In [ ]:
from datasets import Dataset

NUM_SCENARIOS = 3       # Easy / Medium / Hard
EPISODES_PER_SCENARIO = 20  # Roll out each scenario 20 times for variety

samples = []

for scenario_idx in range(NUM_SCENARIOS):
    for episode in range(EPISODES_PER_SCENARIO):
        env = StaleMindEnv(scenario_index=scenario_idx)
        obs = env.reset()
        for step_num in range(1, 11):
            prompt = format_obs_as_prompt(obs, step_num)
            samples.append({
                "prompt": prompt,
                "scenario_index": scenario_idx,
                "step": step_num,
                "drift_active": obs.get("drift_triggered", False),
            })
            # Advance env with a neutral action to get next obs
            obs, _, done, _ = env.step({"type": "ACCEPT", "content": ""})
            if done:
                break

dataset = Dataset.from_list(samples)
dataset = dataset.shuffle(seed=42)

# Split: 90% train, 10% eval
split = dataset.train_test_split(test_size=0.1, seed=42)
train_dataset = split["train"]
eval_dataset  = split["test"]

print(f"Train samples: {len(train_dataset)}")
print(f"Eval samples:  {len(eval_dataset)}")
print(f"\nExample prompt:\n{train_dataset[0]['prompt']}")

## 4. GRPO Reward Function

This is the core of GRPO training. For each prompt, the trainer generates **G completions** (default G=4), scores each with this function, then updates the model to prefer higher-scoring completions relative to the group mean.

Our reward function:
- **+environment reward**: What StaleMind gives for the chosen action (0.0–1.0)
- **+format bonus**: Small reward for valid JSON output (encourages parseable responses)
- **−drift penalty**: Penalizes ACCEPT when drift keywords are clearly present (encourages learning)

In [ ]:
import json, re

def stalemind_reward_fn(prompts: list, completions: list, **kwargs) -> list:
    """
    GRPO reward function.
    Called by GRPOTrainer with batches of (prompt, completion) pairs.
    Returns a list of scalar rewards, one per completion.
    """
    rewards = []
    scenario_indices = kwargs.get("scenario_index", [0] * len(prompts))

    for i, (prompt, completion) in enumerate(zip(prompts, completions)):
        reward = 0.0
        scenario_idx = scenario_indices[i] if i < len(scenario_indices) else 0

        # ── 1. Parse action from model output (THE FIX vs eval notebook) ─────
        action = parse_action_from_completion(completion)

        # ── 2. Run a single env step to get the reward signal ─────────────────
        try:
            env = StaleMindEnv(scenario_index=int(scenario_idx))
            env.reset()
            _, env_reward, _, _ = env.step({"type": action, "content": ""})
            if isinstance(env_reward, dict):
                env_reward = env_reward.get("score", 0.0)
            reward += float(env_reward)
        except Exception as e:
            reward += 0.0  # env error = neutral, don't crash training

        # ── 3. Format bonus: valid JSON with correct keys ─────────────────────
        try:
            match = re.search(r'\{.*?\}', completion, re.DOTALL)
            if match:
                parsed = json.loads(match.group())
                if "action" in parsed and "reasoning" in parsed:
                    reward += 0.1  # small bonus for well-formed output
        except Exception:
            pass

        # ── 4. Drift awareness penalty ────────────────────────────────────────
        # If drift keywords are in the prompt but model still ACCEPTs, penalize
        prompt_lower = prompt.lower()
        drift_present = any(kw in prompt_lower for kw in DRIFT_KEYWORDS)
        if drift_present and action == "ACCEPT":
            reward -= 0.3

        rewards.append(reward)

    return rewards

# Quick sanity check
test_prompts = [train_dataset[0]["prompt"]]
test_completions = ['{"action": "REJECT", "reasoning": "Son needs attention today"}']
test_rewards = stalemind_reward_fn(test_prompts, test_completions, scenario_index=[0])
print(f"Sanity check reward: {test_rewards[0]:.3f}  (should be > 0)")

## 5. GRPO Training

GRPOTrainer handles the RL loop. Key hyperparameters explained inline.

In [ ]:
from trl import GRPOConfig, GRPOTrainer

grpo_config = GRPOConfig(
    # ── Output ────────────────────────────────────────────────────────────────
    output_dir="/content/stalemind-llama3b-grpo",

    # ── Training schedule ─────────────────────────────────────────────────────
    num_train_epochs=3,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,   # effective batch = 16
    learning_rate=5e-6,              # Conservative — RL is sensitive to LR
    warmup_ratio=0.1,
    lr_scheduler_type="cosine",

    # ── GRPO-specific ─────────────────────────────────────────────────────────
    num_generations=4,               # G completions per prompt (higher = better signal, more VRAM)
    max_new_tokens=80,               # Enough for JSON action + reasoning
    temperature=0.8,                 # Some exploration during training
    beta=0.01,                       # KL penalty weight — keeps model close to base

    # ── Memory optimisation ───────────────────────────────────────────────────
    bf16=True,
    gradient_checkpointing=True,
    optim="paged_adamw_8bit",        # 8-bit optimizer to save ~2GB VRAM
    max_grad_norm=0.3,

    # ── Logging & saving ──────────────────────────────────────────────────────
    logging_steps=10,
    save_steps=50,
    eval_steps=50,
    save_total_limit=2,
    report_to="none",                # Change to "wandb" if you want live loss curves
)

trainer = GRPOTrainer(
    model=model,
    reward_funcs=stalemind_reward_fn,
    args=grpo_config,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    tokenizer=tokenizer,
)

print("Starting GRPO training...")
print(f"Total training steps: {len(train_dataset) // (grpo_config.per_device_train_batch_size * grpo_config.gradient_accumulation_steps) * grpo_config.num_train_epochs}")
trainer.train()

## 6. Save the Trained Adapter

Only the LoRA adapter weights are saved (~50MB), not the full 3B model.

In [ ]:
SAVE_PATH = "/content/stalemind-lora-adapter"
model.save_pretrained(SAVE_PATH)
tokenizer.save_pretrained(SAVE_PATH)
print(f"Adapter saved to {SAVE_PATH}")

# Optional: push to HuggingFace Hub
# from huggingface_hub import HfApi
# model.push_to_hub("YOUR_USERNAME/stalemind-llama3b-grpo")
# tokenizer.push_to_hub("YOUR_USERNAME/stalemind-llama3b-grpo")

## 7. Evaluate Trained Model vs Baseline

Run the trained model against StaleMind and compare total reward to the untrained baseline.

In [ ]:
from peft import PeftModel

# Load trained adapter on top of base model
eval_model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID, quantization_config=bnb_config, device_map="auto"
)
eval_model = PeftModel.from_pretrained(eval_model, SAVE_PATH)
eval_model.eval()

def model_fn(prompt: str) -> str:
    """Inference wrapper — takes a prompt string, returns model completion."""
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=512).to("cuda")
    with torch.no_grad():
        outputs = eval_model.generate(
            **inputs,
            max_new_tokens=80,
            do_sample=False,           # Greedy for eval
            temperature=1.0,
            pad_token_id=tokenizer.eos_token_id,
        )
    # Decode only the newly generated tokens (not the prompt)
    new_tokens = outputs[0][inputs["input_ids"].shape[1]:]
    return tokenizer.decode(new_tokens, skip_special_tokens=True)

# ── Run 3 episodes per scenario (Easy / Medium / Hard) ───────────────────────
print("=" * 50)
print("TRAINED MODEL EVALUATION")
print("=" * 50)

scenario_names = ["Easy (drift @ step 3)", "Medium (drift @ step 5)", "Hard (drift @ step 8)"]
all_rewards = []

for scenario_idx, name in enumerate(scenario_names):
    episode_rewards = []
    for trial in range(3):
        reward = run_episode(model_fn, scenario_index=scenario_idx)
        episode_rewards.append(reward)
    avg = sum(episode_rewards) / len(episode_rewards)
    all_rewards.append(avg)
    print(f"{name}: avg reward = {avg:.3f}  (trials: {[round(r,2) for r in episode_rewards]})")

print(f"\nOverall avg reward: {sum(all_rewards)/len(all_rewards):.3f}")
print("(Baseline regex agent scored ~4.16 in your eval notebook — compare against this)")

## 8. Budget Tracker

| Run | Description | Est. Time | Est. Cost |
|-----|-------------|-----------|----------|
| 1 | Baseline eval (no training) | 0.25 hr | $0.30 |
| 2 | GRPO 3 epochs, r=16 | 2.5 hr | $3.00 |
| 3 | GRPO 3 epochs, r=32 (more expressive) | 3 hr | $3.60 |
| 4 | GRPO 5 epochs, best config | 4 hr | $4.80 |
| 5 | Ablation: no drift penalty | 2.5 hr | $3.00 |
| 6 | Ablation: G=8 generations | 3 hr | $3.60 |
| **Total** | | **~15.25 hr** | **~$18.30** |

You have $6.70 buffer for reruns or extended epochs. Staying well within $25.